In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers datasets scikit-learn

In [3]:
!ls /content/drive/MyDrive/disaster_project/disaster_data/

class_weights.pt  label_mapping.json  test.parquet  train.parquet  val.parquet


Load Class Weights (From Drive)

In [4]:
import torch

class_weights = torch.load(
    "/content/drive/MyDrive/disaster_project/disaster_data/class_weights.pt"
)

print(class_weights)

tensor([0.4337, 0.5845, 0.2864, 0.3203, 6.5466, 0.3714, 0.1925, 0.8929, 0.1099,
        0.2619])


In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "/content/drive/MyDrive/disaster_project/disaster_data/train.parquet",
        "validation": "/content/drive/MyDrive/disaster_project/disaster_data/val.parquet",
        "test": "/content/drive/MyDrive/disaster_project/disaster_data/test.parquet",
    }
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tweet_text', 'class_label', 'low_info', 'label'],
        num_rows: 53531
    })
    validation: Dataset({
        features: ['tweet_text', 'class_label', 'low_info', 'label'],
        num_rows: 7793
    })
    test: Dataset({
        features: ['tweet_text', 'class_label', 'low_info', 'label'],
        num_rows: 15160
    })
})

Load Label Mapping

In [6]:
import json

with open("/content/drive/MyDrive/disaster_project/disaster_data/label_mapping.json") as f:
    label_mapping = json.load(f)

label2id = label_mapping["label2id"]
id2label = label_mapping["id2label"]

Load Tokenizer & Model

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=10,
    id2label=id2label,
    label2id=label2id
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenize Dataset

In [8]:
def tokenize_function(examples):
    return tokenizer(
        examples["tweet_text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

dataset = dataset.map(tokenize_function, batched=True)

dataset = dataset.remove_columns(["tweet_text"])
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch")

Map:   0%|          | 0/53531 [00:00<?, ? examples/s]

Map:   0%|          | 0/7793 [00:00<?, ? examples/s]

Map:   0%|          | 0/15160 [00:00<?, ? examples/s]

Training Arguments

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/disaster_project/disaster_models/roberta_weighted",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
)

Define WeightedTrainer

In [16]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [17]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, predictions, average="macro")
    return {"macro_f1": macro_f1}

Initialize Trainer

In [18]:
trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

Change Output Directory

In [19]:
output_dir="/content/drive/MyDrive/disaster_project/disaster_models/roberta_weighted"

Train

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.773542,0.679293,0.741522
2,0.557839,0.633639,0.747538
3,0.438426,0.666249,0.750787


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=10038, training_loss=0.5899356249961085, metrics={'train_runtime': 3950.0975, 'train_samples_per_second': 40.655, 'train_steps_per_second': 2.541, 'total_flos': 1.0564207187885568e+16, 'train_loss': 0.5899356249961085, 'epoch': 3.0})

In [21]:
trainer.evaluate(dataset["test"])

{'eval_loss': 0.677177369594574,
 'eval_macro_f1': 0.757698183343688,
 'eval_runtime': 114.2696,
 'eval_samples_per_second': 132.669,
 'eval_steps_per_second': 8.296,
 'epoch': 3.0}

In [22]:
from sklearn.metrics import classification_report

predictions = trainer.predict(dataset["test"])
preds = predictions.predictions.argmax(axis=1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=list(label2id.keys())))

                                        precision    recall  f1-score   support

                    caution_and_advice       0.63      0.80      0.71      1070
      displaced_people_and_evacuations       0.83      0.94      0.88       790
     infrastructure_and_utility_damage       0.79      0.85      0.82      1617
                injured_or_dead_people       0.91      0.95      0.93      1447
               missing_or_found_people       0.76      0.82      0.79        72
                      not_humanitarian       0.63      0.66      0.64      1245
            other_relevant_information       0.63      0.44      0.52      2407
              requests_or_urgent_needs       0.52      0.70      0.60       521
rescue_volunteering_or_donation_effort       0.88      0.86      0.87      4219
                  sympathy_and_support       0.83      0.83      0.83      1772

                              accuracy                           0.77     15160
                             macro avg